In [2]:
import pandas as pd
import numpy as np

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
apartment_for_rent_classified = fetch_ucirepo(id=555) 
  
# data (as pandas dataframes) 
df = apartment_for_rent_classified.data.features 
  
# metadata 
print(apartment_for_rent_classified.metadata) 
  
# variable information 
print(apartment_for_rent_classified.variables) 


{'uci_id': 555, 'name': 'Apartment for Rent Classified', 'repository_url': 'https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified', 'data_url': 'https://archive.ics.uci.edu/static/public/555/data.csv', 'abstract': 'This is a dataset of classified for apartments for rent in USA.\n', 'area': 'Business', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 10000, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': [], 'target_col': None, 'index_col': ['id'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C5X623', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "The dataset contains of 10'000 or 100'000 rows and of 22 columns The data has been cleaned in the way that \r\ncolumn price and square_feet never is empty but the dataset is saved as it was created.\r\n\r\n

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0,5,6,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


### Notes/Ideas, Process?

- Create repo, do the whole GIT thing
- Data exploration?
    - Summary stats, for price primarily
    - Visuals, general graphs, histograms, pairplots, etc.
    - Correlations, relationships
    - Basic linear models?
- Strategy for training, testing
    - Initially split into train and test datasets (80/20? 75/25?)
    - Run CV on training data
    - Evaluate models on test data
- Establish Evaluation Strategy
    - RMSE
    - R-squared
    - Mean Absolute Error?
    - Residual analysis?
    - Sensitivity analysis?
- Establish CV, hyperparameter tuning Strategy
    - Custom function for gridsearchCV?? or Random search CV?
    - Maybe different approaches for different model groups
- Simpler experimental ML models - regression based
    - Decision Trees
    - KNN - SCALING
    - SVR ??? - SCALING
    - Regularized regression - SCALING
- Ensemble methods
    - Random Forest
    - XGBoost
    - Custom ensemble
- Model Evaluation and comparison
    - Tables, visuals


- Make two helper functions?
    - One pipeline for training and scaling,
    - One pipeline for CV, tuning, and evaluation


### Crucial Best Practices for This Workflow

To make sure this workflow is mathematically sound, keep these three execution rules in mind:

- **Fit Transformers ONLY on the Training Data**: If you are scaling features (e.g., StandardScaler), computing the mean, or imputing missing values, you must .fit() your scaler only on the training data. You then .transform() both the training and test data. Fitting a scaler on the full dataset before splitting leaks the global mean and variance into your test set.

- **Keep Random Seeds Consistent**: When comparing multiple models using CV on the training set, ensure you lock the random state or use the exact same cross-validation splits (e.g., passing the same KFold object to all models). This guarantees that Model A and Model B are being evaluated on the exact same subsets of data.

- **Never Go Backward**: If your final test set score is disappointing, do not go back and tweak your hyperparameters to make the test score better. If you do, your test set is no longer a true test set; it has become part of your tuning process, and your final metrics will be overly optimistic.

# To Do: 

- NOTE: month looks suspect, probably don't want to use.
- Create XGBoost Model
- Conduct statistical tests in new notebook? could be some interesting insights
    - Differences between states? Impact of certain variables?

In [4]:
df.head()

,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,USD,No,Thumbnail,Cats,...,2195,Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1.577360e+09
1,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,USD,No,Thumbnail,"Cats,Dogs",...,1250,Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1.577360e+09
2,housing/rent/apartment,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,USD,No,Thumbnail,NaN,...,1395,Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1.577360e+09
3,housing/rent/apartment,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,USD,No,Thumbnail,"Cats,Dogs",...,1600,Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1.577360e+09
4,housing/rent/apartment,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,USD,No,Thumbnail,"Cats,Dogs",...,975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1.577360e+09


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99826 entries, 0 to 99825
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   category       99826 non-null  object 
 1   title          99826 non-null  object 
 2   body           99826 non-null  object 
 3   amenities      83749 non-null  object 
 4   bathrooms      99760 non-null  object 
 5   bedrooms       99699 non-null  object 
 6   currency       99822 non-null  object 
 7   fee            99823 non-null  object 
 8   has_photo      99823 non-null  object 
 9   pets_allowed   39192 non-null  object 
 10  price          99821 non-null  float64
 11  price_display  99820 non-null  object 
 12  price_type     99823 non-null  object 
 13  square_feet    99823 non-null  object 
 14  address        7946 non-null   object 
 15  cityname       99521 non-null  object 
 16  state          99521 non-null  object 
 17  latitude       99797 non-null  float64
 18  longit

In [6]:
df.isna().sum()

category             0
title                0
body                 0
amenities        16077
bathrooms           66
bedrooms           127
currency             4
fee                  3
has_photo            3
pets_allowed     60634
price                5
price_display        6
price_type           3
square_feet          3
address          91880
cityname           305
state              305
latitude            29
longitude           31
source               6
time                 6
dtype: int64

Drop Null values applied to columns with small numbers of Null values.

In [7]:
cols_to_drop = df.columns[df.isna().sum() <= 500]

df = df.dropna(subset=cols_to_drop)

In [8]:
# Large amounts of null values in both pets_allowed, and address
# Likely just remove both
# Can I produde other variables from amenities? e.g. 'has_ac', 'has_pool', etc.

df.isna().sum()

category             0
title                0
body                 0
amenities        15871
bathrooms            0
bedrooms             0
currency             0
fee                  0
has_photo            0
pets_allowed     60321
price                0
price_display        0
price_type           0
square_feet          0
address          91508
cityname             0
state                0
latitude             0
longitude            0
source               0
time                 0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99335 entries, 0 to 99825
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   category       99335 non-null  object 
 1   title          99335 non-null  object 
 2   body           99335 non-null  object 
 3   amenities      83464 non-null  object 
 4   bathrooms      99335 non-null  object 
 5   bedrooms       99335 non-null  object 
 6   currency       99335 non-null  object 
 7   fee            99335 non-null  object 
 8   has_photo      99335 non-null  object 
 9   pets_allowed   39014 non-null  object 
 10  price          99335 non-null  float64
 11  price_display  99335 non-null  object 
 12  price_type     99335 non-null  object 
 13  square_feet    99335 non-null  object 
 14  address        7827 non-null   object 
 15  cityname       99335 non-null  object 
 16  state          99335 non-null  object 
 17  latitude       99335 non-null  float64
 18  longitude  

In [10]:
df['price'].describe()

count    99335.000000
mean      1525.818402
std        898.358782
min        100.000000
25%       1014.000000
50%       1350.000000
75%       1795.000000
max      52500.000000
Name: price, dtype: float64

In [11]:
# Is there any difference between price, price_display, and currency? 
# Currency not needed, USD for all.
# price_display same as price, need to remove one to avoid leakage.

df[['price', 'currency', 'price_display']].head()

,price,currency,price_display
0,2195.0,USD,2195
1,1250.0,USD,1250
2,1395.0,USD,1395
3,1600.0,USD,1600
4,975.0,USD,975


In [12]:
# Weekly vs monthly will make the amounts very different, remove those with type Weekly.

df['price_type'].value_counts()

price_type
Monthly    99332
Weekly         3
Name: count, dtype: int64

In [13]:
df = df[df['price_type'] == 'Monthly']

In [14]:
df_reduced = df.drop(
    columns=['price_display', 'price_type', 'currency', 'address', 'category', 'source']
    )

df_reduced.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99332 entries, 0 to 99825
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         99332 non-null  object 
 1   body          99332 non-null  object 
 2   amenities     83461 non-null  object 
 3   bathrooms     99332 non-null  object 
 4   bedrooms      99332 non-null  object 
 5   fee           99332 non-null  object 
 6   has_photo     99332 non-null  object 
 7   pets_allowed  39013 non-null  object 
 8   price         99332 non-null  float64
 9   square_feet   99332 non-null  object 
 10  cityname      99332 non-null  object 
 11  state         99332 non-null  object 
 12  latitude      99332 non-null  float64
 13  longitude     99332 non-null  float64
 14  time          99332 non-null  float64
dtypes: float64(4), object(11)
memory usage: 12.1+ MB


In [15]:
df_reduced['state'].value_counts()

state
TX    11258
CA    10284
VA     8302
NC     6306
CO     6285
FL     5781
MD     5305
MA     5037
OH     4908
GA     4790
NJ     4494
NV     2815
WA     2617
AZ     2378
LA     1354
MO     1202
PA     1134
TN     1114
IL     1032
NE     1019
KY      994
OK      959
KS      911
SC      908
UT      809
ND      740
NH      733
MI      709
NY      651
AR      598
MN      579
CT      513
IN      503
WI      428
IA      371
AL      352
OR      277
VT      125
RI      119
MS      107
ID       96
MT       88
SD       86
DC       83
AK       57
HI       31
ME       30
NM       24
WY       16
WV       13
DE        7
Name: count, dtype: int64

In [16]:
# Create dummy vars from state, but add the state column back for later analysis

df_add_state = pd.get_dummies(df_reduced, columns=['state'], dtype='int')

df_add_state['state'] = df_reduced['state']

df_add_state.head()

,title,body,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,price,square_feet,...,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,state
0,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,No,Thumbnail,Cats,2195.0,542,...,0,0,0,0,0,0,0,0,0,CA
1,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,No,Thumbnail,"Cats,Dogs",1250.0,1500,...,0,0,0,1,0,0,0,0,0,VA
2,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,No,Thumbnail,NaN,1395.0,1650,...,0,0,0,0,0,0,0,0,0,NC
3,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,No,Thumbnail,"Cats,Dogs",1600.0,820,...,0,0,0,0,0,0,0,0,0,CA
4,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,No,Thumbnail,"Cats,Dogs",975.0,624,...,0,0,0,0,0,0,0,0,0,NM


In [17]:
df_add_state['state_TX'].sum()

np.int64(11258)

In [18]:
# Very large clustering in Sep, Feb, and Dec. Seems odd, I will likely not use month.

df_add_state['date'] = pd.to_datetime(df_add_state['time'].astype(float), unit='s')

df_add_state['month'] = df_add_state['date'].dt.month

df_add_state['month'].value_counts()

month
9     43339
2     31208
12    23260
7       464
8       302
6       168
3       132
5       119
1       119
4       107
10       65
11       49
Name: count, dtype: int64

In [19]:
# Drop 'time' now that it is redundant

df_add_state.drop(columns=['time'])

,title,body,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,price,square_feet,...,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,state,date,month
0,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,No,Thumbnail,Cats,2195.0,542,...,0,0,0,0,0,0,0,CA,2019-12-26 11:39:15,12
1,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,No,Thumbnail,"Cats,Dogs",1250.0,1500,...,0,1,0,0,0,0,0,VA,2019-12-26 11:39:00,12
2,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,No,Thumbnail,NaN,1395.0,1650,...,0,0,0,0,0,0,0,NC,2019-12-26 11:38:52,12
3,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,No,Thumbnail,"Cats,Dogs",1600.0,820,...,0,0,0,0,0,0,0,CA,2019-12-26 11:38:50,12
4,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,No,Thumbnail,"Cats,Dogs",975.0,624,...,0,0,0,0,0,0,0,NM,2019-12-26 11:38:28,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99821,Houston - superb Apartment nearby fine dining,"Redefining urban living, in southeast Houston,...","Gym,Parking,Patio/Deck,Playground,Storage,Wood...",1.0,1.0,No,Yes,NaN,780.0,605,...,0,0,0,0,0,0,0,TX,2018-12-07 12:29:30,12
99822,The Best of the Best in the City of Jacksonvil...,Courtney Manor Apartments offer the best of ev...,"AC,Cable or Satellite,Clubhouse,Dishwasher,Gym...",2.0,2.0,No,Yes,"Cats,Dogs",813.0,921,...,0,0,0,0,0,0,0,FL,2018-12-07 12:29:10,12
99823,A great & large One BR apartment. Pet OK!,"Fully remodeled, new floor, kitchen cabinet, s...","Garbage Disposal,Refrigerator",1.0,1.0,No,Yes,"Cats,Dogs",1325.0,650,...,0,0,0,0,0,0,0,CA,2018-12-07 12:28:49,12
99824,"The Crest offers studio, 1, 2 & Three BR homes...","Amenities include a fitness facilities, swimmi...","Gym,Pool",1.0,1.0,No,Yes,"Cats,Dogs",931.0,701,...,0,0,0,0,0,0,0,NC,2018-12-07 12:27:50,12


In [20]:
# List of features to add. Not exaustive, manually selected after visual inspection 
# of the data. 

amenities_list = [
    'pool',
    'dishwasher',
    'gym',
    'parking',
    'fireplace',
    'patio/deck',
    'storage'
]

for a in amenities_list: 
    df_add_state[a] = 0

df_add_state.head()

,title,body,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,price,square_feet,...,state,date,month,pool,dishwasher,gym,parking,fireplace,patio/deck,storage
0,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,No,Thumbnail,Cats,2195.0,542,...,CA,2019-12-26 11:39:15,12,0,0,0,0,0,0,0
1,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,No,Thumbnail,"Cats,Dogs",1250.0,1500,...,VA,2019-12-26 11:39:00,12,0,0,0,0,0,0,0
2,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,No,Thumbnail,NaN,1395.0,1650,...,NC,2019-12-26 11:38:52,12,0,0,0,0,0,0,0
3,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,No,Thumbnail,"Cats,Dogs",1600.0,820,...,CA,2019-12-26 11:38:50,12,0,0,0,0,0,0,0
4,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,No,Thumbnail,"Cats,Dogs",975.0,624,...,NM,2019-12-26 11:38:28,12,0,0,0,0,0,0,0


In [21]:
# Loop through amenities list and check if each row contains that text.

for a in amenities_list:
    matches = df_add_state['amenities'].str.lower().str.contains(a, na=False)
    df_add_state.loc[matches, a] = 1

df_add_state.head()


,title,body,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,price,square_feet,...,state,date,month,pool,dishwasher,gym,parking,fireplace,patio/deck,storage
0,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,No,Thumbnail,Cats,2195.0,542,...,CA,2019-12-26 11:39:15,12,0,0,0,0,0,0,0
1,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,No,Thumbnail,"Cats,Dogs",1250.0,1500,...,VA,2019-12-26 11:39:00,12,0,0,0,0,0,0,0
2,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,No,Thumbnail,NaN,1395.0,1650,...,NC,2019-12-26 11:38:52,12,0,0,0,0,0,0,0
3,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,No,Thumbnail,"Cats,Dogs",1600.0,820,...,CA,2019-12-26 11:38:50,12,0,0,0,0,0,0,0
4,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,No,Thumbnail,"Cats,Dogs",975.0,624,...,NM,2019-12-26 11:38:28,12,0,0,0,0,0,0,0


In [22]:
# Turn pets_allowed into binary variable

df_add_state['pets_allowed'] = np.where(df_add_state['pets_allowed'].notna(), 1, 0)



In [23]:
# Turn fee into binary variable

df_add_state['fee'] = np.where(df_add_state['fee'] == 'Yes', 1, 0)

In [24]:
df_add_state.columns

Index(['title', 'body', 'amenities', 'bathrooms', 'bedrooms', 'fee',
       'has_photo', 'pets_allowed', 'price', 'square_feet', 'cityname',
       'latitude', 'longitude', 'time', 'state_AK', 'state_AL', 'state_AR',
       'state_AZ', 'state_CA', 'state_CO', 'state_CT', 'state_DC', 'state_DE',
       'state_FL', 'state_GA', 'state_HI', 'state_IA', 'state_ID', 'state_IL',
       'state_IN', 'state_KS', 'state_KY', 'state_LA', 'state_MA', 'state_MD',
       'state_ME', 'state_MI', 'state_MN', 'state_MO', 'state_MS', 'state_MT',
       'state_NC', 'state_ND', 'state_NE', 'state_NH', 'state_NJ', 'state_NM',
       'state_NV', 'state_NY', 'state_OH', 'state_OK', 'state_OR', 'state_PA',
       'state_RI', 'state_SC', 'state_SD', 'state_TN', 'state_TX', 'state_UT',
       'state_VA', 'state_VT', 'state_WA', 'state_WI', 'state_WV', 'state_WY',
       'state', 'date', 'month', 'pool', 'dishwasher', 'gym', 'parking',
       'fireplace', 'patio/deck', 'storage'],
      dtype='object')

In [25]:
# Fix datatype inconsistency in 'bedrooms'

df_add_state['bedrooms'] = df_add_state['bedrooms'].astype(int)

In [26]:
# Fix datatype inconsistency in 'bathrooms'

df_add_state['bathrooms'] = df_add_state['bathrooms'].astype(float)

In [27]:
# Fix datatype inconsistency in 'square_feet'

df_add_state['square_feet'] = df_add_state['square_feet'].astype(int)

In [28]:
df_add_state.head(10)

,title,body,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,price,square_feet,...,state,date,month,pool,dishwasher,gym,parking,fireplace,patio/deck,storage
0,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1.0,1,0,Thumbnail,1,2195.0,542,...,CA,2019-12-26 11:39:15,12,0,0,0,0,0,0,0
1,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,0,Thumbnail,1,1250.0,1500,...,VA,2019-12-26 11:39:00,12,0,0,0,0,0,0,0
2,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2.0,3,0,Thumbnail,0,1395.0,1650,...,NC,2019-12-26 11:38:52,12,0,0,0,0,0,0,0
3,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1.0,2,0,Thumbnail,1,1600.0,820,...,CA,2019-12-26 11:38:50,12,0,0,0,0,0,0,0
4,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1.0,1,0,Thumbnail,1,975.0,624,...,NM,2019-12-26 11:38:28,12,0,0,0,0,0,0,0
5,Two BR 7801 Marble NE,"This unit is located at 7801 Marble NE, Albuqu...",NaN,1.5,2,0,Thumbnail,1,1250.0,965,...,NM,2019-12-26 11:38:28,12,0,0,0,0,0,0,0
6,Two BR 5 Salt Marsh Quay Apartment H,This unit is located at five Salt Marsh Quay A...,NaN,2.0,2,0,Thumbnail,0,1600.0,1120,...,VA,2019-12-26 11:37:41,12,0,0,0,0,0,0,0
7,Two BR 11280 W. 20th Ave.,"This unit is located at 11280 W. 20th Ave., La...",NaN,2.0,2,0,Thumbnail,1,1300.0,947,...,CO,2019-12-26 11:37:27,12,0,0,0,0,0,0,0
8,Two BR 1427 Lewis Street,"This unit is located at 1427 Lewis Street, Cha...",NaN,1.0,2,0,Thumbnail,1,795.0,600,...,WV,2019-12-26 11:37:19,12,0,0,0,0,0,0,0
9,Two BR 10201 Remmet Avenue,"This unit is located at 10201 Remmet Avenue, C...",NaN,2.0,2,0,Thumbnail,0,2150.0,1005,...,CA,2019-12-26 11:36:44,12,0,0,0,0,0,0,0


In [29]:
# has_photo variable has multiple options, one for thumbnail and one for photo,
# so creating dummies for both.

df_add_state = pd.get_dummies(df_add_state, columns=['has_photo'], dtype=int)

In [30]:
# Renaming for clarity.

df_add_state = df_add_state.drop(columns='has_photo_No')
df_add_state = df_add_state.rename(
    columns={
        'has_photo_Thumbnail': 'has_photo_thumbnail_only', 
        'has_photo_Yes': 'has_photo'})

df_add_state.head()

,title,body,amenities,bathrooms,bedrooms,fee,pets_allowed,price,square_feet,cityname,...,month,pool,dishwasher,gym,parking,fireplace,patio/deck,storage,has_photo_thumbnail_only,has_photo
0,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1.0,1,0,1,2195.0,542,Redondo Beach,...,12,0,0,0,0,0,0,0,1,0
1,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,0,1,1250.0,1500,Newport News,...,12,0,0,0,0,0,0,0,1,0
2,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2.0,3,0,0,1395.0,1650,Raleigh,...,12,0,0,0,0,0,0,0,1,0
3,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1.0,2,0,1,1600.0,820,Vacaville,...,12,0,0,0,0,0,0,0,1,0
4,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1.0,1,0,1,975.0,624,Albuquerque,...,12,0,0,0,0,0,0,0,1,0


In [31]:
df_outliers = (
    df_add_state[df_add_state['price'] >= 10000]
    )

df_outliers['price'].describe()

count       77.000000
mean     16796.285714
std       8395.764386
min      10000.000000
25%      11500.000000
50%      13500.000000
75%      17000.000000
max      52500.000000
Name: price, dtype: float64

In [32]:
len(df_add_state)

99332

In [33]:
# Most expensive entry is clearly a mistake - Studio apartment, body text says
# that rent is $500 a month.


df_add_state = df_add_state[df_add_state['price'] != 52500]


In [34]:
# largest enry by sqft also a mistake, 12000 ft studio for $565?

df_add_state = df_add_state[df_add_state['square_feet'] != 12000]

In [35]:
# # Add new metrics, bedroom and bathroom to square foot ratios
# # Increment bedrooms by 1 to avoid dividing by 0

# df_add_state['sqft_per_bedroom'] = (
#     df_add_state['square_feet'] / (df_add_state['bedrooms'] + 1)
# )

# df_add_state['sqft_per_bathroom'] = (
#     df_add_state['square_feet'] / df_add_state['bathrooms']
# )

In [231]:
df_add_state.columns

Index(['title', 'body', 'amenities', 'bathrooms', 'bedrooms', 'fee',
       'pets_allowed', 'price', 'square_feet', 'cityname', 'latitude',
       'longitude', 'time', 'state_AK', 'state_AL', 'state_AR', 'state_AZ',
       'state_CA', 'state_CO', 'state_CT', 'state_DC', 'state_DE', 'state_FL',
       'state_GA', 'state_HI', 'state_IA', 'state_ID', 'state_IL', 'state_IN',
       'state_KS', 'state_KY', 'state_LA', 'state_MA', 'state_MD', 'state_ME',
       'state_MI', 'state_MN', 'state_MO', 'state_MS', 'state_MT', 'state_NC',
       'state_ND', 'state_NE', 'state_NH', 'state_NJ', 'state_NM', 'state_NV',
       'state_NY', 'state_OH', 'state_OK', 'state_OR', 'state_PA', 'state_RI',
       'state_SC', 'state_SD', 'state_TN', 'state_TX', 'state_UT', 'state_VA',
       'state_VT', 'state_WA', 'state_WI', 'state_WV', 'state_WY', 'state',
       'date', 'month', 'pool', 'dishwasher', 'gym', 'parking', 'fireplace',
       'patio/deck', 'storage', 'has_photo_thumbnail_only', 'has_photo'],
    

In [66]:
import os
from pathlib import Path

root = Path.cwd()

In [233]:
df_add_state.to_csv(root / 'cleaned_apartment_data.csv', index=False)


### Remove outliers, create new csv

In [61]:
df_high_price = df_add_state[df_add_state['price'] >= 5000]

In [62]:
df_high_price['price'].describe()

count      497.000000
mean      7972.939638
std       4709.286559
min       5000.000000
25%       5500.000000
50%       6500.000000
75%       8250.000000
max      40000.000000
Name: price, dtype: float64

In [ ]:
# Only ~500 entries with price over $5000, removing them to see impact on predictions

df_outliers_removed = df_add_state[df_add_state['price'] < 5000]

In [65]:
df_outliers_removed['price'].describe()

count    98833.000000
mean      1492.827750
std        680.398011
min        100.000000
25%       1010.000000
50%       1350.000000
75%       1784.000000
max       4999.000000
Name: price, dtype: float64

In [67]:
df_outliers_removed.to_csv(root / 'cleaned_apartment_data_outliers_removed.csv', index=False)